In [ ]:

import json
import pandas as pd
import traceback


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace

import PyPDF2

In [ ]:
load_dotenv()


In [ ]:
llm = HuggingFaceEndpoint(repo_id = "Qwen/Qwen3-Coder-Next",
                          huggingfacehub_api_token=os.getenv(
        "HUGGINGFACE_HUB_API"
    ),
    task = "conversational",
    temperature = 0.5,
    max_new_tokens=4000
                 )


chat = ChatHuggingFace(llm=llm)

In [ ]:
response_format = """
{
  "quiz": {
    "subject": "string",
    "difficulty_level": "Easy | Medium | Hard",
    "total_questions": 0,
    "questions": [
      {
        "question_number": 1,
        "question": "string",
        "options": {
          "A": "string",
          "B": "string",
          "C": "string",
          "D": "string"
        },
        "correct_answer": "A",
        
      }
    ]
  }
}
"""

In [ ]:


Template = """
You are an expert quiz generator.

Generate a quiz based on the following requirements:

Text:
{text}

Subject:
{subject}

Number of Questions:
{total_questions}

Difficulty Level:
{difficulty_level}

Instructions:
1. Generate exactly {total_questions} questions.
2. Questions must be based only on the provided text.
3. Match the difficulty level: {difficulty_level}.
4. Keep the quiz relevant to the subject: {subject}.
5. Include a mix of conceptual and factual questions where appropriate.
6. Avoid duplicate or ambiguous questions.
7. Return ONLY valid JSON.
8. Do not use markdown.
9. Do not add explanations.
10. Ensure all strings are properly escaped.
11. Generate the output strictly in this format:

{response_format}


"""




In [ ]:
quiz_generation_prompt = ChatPromptTemplate.from_template(
    #Input_variables = ["Text", "total_questions", "difficulty_level", "subject" , "response_format"],
    template = Template
)

In [ ]:

chain =  quiz_generation_prompt | chat

In [ ]:
file_path = r"C:\Users\abhia\mcqgenerator\text.txt"

with open(file_path, 'r') as file:
    text =file.read()

In [ ]:
subject = "HIstory"
total_questions = 5
difficulty_level = "Medium"

In [ ]:
print(type(chain))
print(chain)

In [ ]:
response = chain.invoke({
    "text": text,
    "subject": subject,
    "total_questions": total_questions,
    "difficulty_level": difficulty_level,
    "response_format": response_format
})


In [ ]:
print(response)

In [ ]:

print(response)

In [ ]:
raw = (
    response.content
    .replace("```json", "")
    .replace("```", "")
    .strip()
)

print(raw)


In [ ]:
response_json = json.loads(response.content)